In [1]:
%%writefile app.py
%%writefile app.py
import streamlit as st
import pandas as pd

st.set_page_config(layout="wide", page_title="Pipeline Optimizer")

st.title("🏭 Factory 2.0: Agile Opportunity Pipeline Matrix")
st.write("Adjust global values or fill in local variables to dynamically calculate project priority.")

# 1. Global Inputs (Sidebar)
st.sidebar.header("🌍 Global Financial Multipliers")
global_fte_cost = st.sidebar.number_input("Global FTE Cost (SGD/Yr)", value=80000, step=5000)
global_scrap_cost_pool = st.sidebar.number_input("Global Plant Scrap Baseline (SGD/Yr)", value=1500000, step=50000)
global_hour_value = st.sidebar.number_input("Value of 1% Schedule Adherence (SGD/Yr)", value=120000, step=10000)

# Weights
st.sidebar.subheader("🎯 Prioritization Weighting")
w_roi = st.sidebar.slider("Financial ROI Weight (%)", 0, 100, 40) / 100
w_trl = st.sidebar.slider("Technical Readiness (TRL) Weight (%)", 0, 100, 30) / 100
w_time = st.sidebar.slider("Speed to Value Weight (%)", 0, 100, 15) / 100
w_scale = st.sidebar.slider("Scalability Footprint Weight (%)", 0, 100, 15) / 100

# 2. Local Fill-in-the-Blanks Table Template
st.subheader("📝 Opportunity Pipeline Template Workspace")

template_data = {
    "Opportunity": [
        "A. Smart visual inspection for manual quality checks",
        "B. Robotics-enabled repetitive manual task automation",
        "C. AI-based production scheduling support",
        "D. Autonomous material transport between process areas",
        "E. AR/VR digital training and remote support",
        "F. Low-TRL sensor for real-time process quality monitoring"
    ],
    "TRL": [6, 4, 5, 7, 6, 3],
    "Est_Cost_SGD": [450000, 700000, 300000, 550000, 150000, 250000],
    "Benefit_FTE": [1.5, 2.0, 0.0, 1.0, 0.0, 0.0],
    "Benefit_Quality_Scrap_Pct": [5.0, 0.0, 0.0, 0.0, 0.0, 10.0],
    "Benefit_Schedule_Improvement_Pct": [0.0, 0.0, 5.0, 0.0, 0.0, 0.0],
    "Benefit_Unquantified_Value": [0, 0, 0, 0, 1, 1],
    "Time_Months": [9, 12, 6, 8, 4, 18],
    "Main_Risk": [
        "Integration with current process", "Low maturity / process variability",
        "Data quality / user adoption / IT integration", "EHS / layout constraints",
        "User adoption, OT hardware validation", "Technology maturity"
    ],
    "Scale_Up_Sites": [4, 6, 3, 5, 8, 4]
}

base_df = pd.DataFrame(template_data)

edited_df = st.data_editor(
    base_df,
    column_config={
        "Opportunity": st.column_config.TextColumn("1) Opportunity", disabled=True),
        "TRL": st.column_config.NumberColumn("2) TRL", min_value=1, max_value=9, step=1),
        "Est_Cost_SGD": st.column_config.NumberColumn("3) Est Cost (SGD)", min_value=0),
        "Benefit_FTE": st.column_config.NumberColumn("4.1) Benefit: FTE Savings", min_value=0.0, step=0.5),
        "Benefit_Quality_Scrap_Pct": st.column_config.NumberColumn("4.2) Benefit: Scrap Reduc. (%)", min_value=0.0, max_value=100.0, step=0.5),
        "Benefit_Schedule_Improvement_Pct": st.column_config.NumberColumn("4.3) Benefit: Schedule Improv. (%)", min_value=0.0, max_value=100.0, step=0.5),
        "Benefit_Unquantified_Value": st.column_config.SelectboxColumn("4.4) Unquantified Strategic Value?", options=[0, 1]),
        "Time_Months": st.column_config.NumberColumn("5) Time (Months)", min_value=1),
        "Main_Risk": st.column_config.TextColumn("6) Main Risk"),
        "Scale_Up_Sites": st.column_config.NumberColumn("7) Scale-up (Sites)", min_value=1),
    },
    hide_index=True, use_container_width=True
)

# 3. Calculation Engine
edited_df["Calculated_Annual_Benefit_Per_Site"] = (
    (edited_df["Benefit_FTE"] * global_fte_cost) + 
    ((edited_df["Benefit_Quality_Scrap_Pct"] / 100) * global_scrap_cost_pool) +
    ((edited_df["Benefit_Schedule_Improvement_Pct"]) * global_hour_value) +
    (edited_df["Benefit_Unquantified_Value"] * 25000)
)
edited_df["Total_Portfolio_Benefit"] = edited_df["Calculated_Annual_Benefit_Per_Site"] * edited_df["Scale_Up_Sites"]
edited_df["Dynamic_ROI_Ratio"] = edited_df["Total_Portfolio_Benefit"] / edited_df["Est_Cost_SGD"].replace(0, 1)

# Dynamic Normalized Scoring Rules (1-5)
edited_df["Score_ROI"] = pd.qcut(edited_df["Dynamic_ROI_Ratio"], q=6, labels=False, duplicates='drop') + 1
edited_df["Score_TRL"] = (edited_df["TRL"] / 9) * 5
edited_df["Score_Time"] = ((24 - edited_df["Time_Months"]).clip(lower=1) / 24) * 5
edited_df["Score_Scale"] = (edited_df["Scale_Up_Sites"] / 10).clip(upper=1) * 5

# Weighted Index Summation
edited_df["Final_Priority_Score"] = (
    (edited_df["Score_ROI"] * w_roi) + (edited_df["Score_TRL"] * w_trl) + 
    (edited_df["Score_Time"] * w_time) + (edited_df["Score_Scale"] * w_scale)
)

final_ranked_df = edited_df.sort_values(by="Final_Priority_Score", ascending=False).reset_index(drop=True)

# 4. View Outputs
st.markdown("---")
st.subheader("📊 Dynamically Optimized Project Ranking Pipeline")
st.success(f"🚀 **Real-Time Decision Directive:** Immediately activate **{final_ranked_df.iloc[0]['Opportunity']}**.")

st.dataframe(
    final_ranked_df[["Opportunity", "Final_Priority_Score", "Total_Portfolio_Benefit", "Dynamic_ROI_Ratio", "TRL", "Time_Months", "Scale_Up_Sites"]],
    use_container_width=True, hide_index=True
)

Writing app.py


In [3]:
!python3 -m streamlit run app.py

^C
  Stopping...
